# Import Libraries

In [15]:
# Core libraries
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Data preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Machine learning models (classification focus)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression

# Evaluation metrics
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score


# Load Dataset

In [16]:
# Load dataset 
data = pd.read_csv('../data/bank_customers_large.csv')
print("Dataset loaded successfully. Shape:", data.shape)

Dataset loaded successfully. Shape: (50000, 16)


# Basic Data Cleaning
Handle missing values by converting numeric-like objects, imputing medians for numeric data, and modes for categorical data.


In [17]:
# Basic Data Cleaning and Missing Value Handling
for col in data.columns:
    if data[col].dtype == 'object':
        # Attempt to convert object columns to numeric
        numeric_col = pd.to_numeric(data[col], errors='coerce')

        if numeric_col.notna().any():
            # If conversion is valid, treat as numeric and fill missing with median
            data[col] = numeric_col.fillna(numeric_col.median())
        else:
            # Otherwise, treat as categorical and fill missing with mode
            mode_val = data[col].mode(dropna=True)
            if not mode_val.empty:
                data[col] = data[col].fillna(mode_val.iloc[0])
            else:
                # Fallback for columns with all missing values
                data[col] = data[col].fillna('Unknown')
    else:
        # Fill missing numeric values with median
        if data[col].isna().any():
            data[col] = data[col].fillna(data[col].median())



# Advanced Feature Engineering
Create interaction features, tenure groups, customer segments, and encode categorical variables for modeling.


In [18]:
# Advanced Feature Engineering

# Create ratio and interaction features
data['Balance_to_Salary'] = data['Balance'] / (data['EstimatedSalary'] + 1)
data['Balance_Tenure'] = data['Balance'] * data['Tenure']
data['Products_Active'] = data['NumOfProducts'] * data['IsActiveMember']

# Create tenure-based groups
data['TenureGroup'] = pd.cut(
    data['Tenure'], bins=[0, 2, 5, 10, 20], labels=[0, 1, 2, 3], include_lowest=True
)
data['TenureGroup'] = data['TenureGroup'].cat.codes.replace(-1, np.nan)
data['TenureGroup'] = data['TenureGroup'].fillna(data['TenureGroup'].median())

# Customer segmentation using K-Means clustering
kmeans = KMeans(n_clusters=4, random_state=42)
data['Segment'] = kmeans.fit_predict(data[['Balance', 'NumOfProducts', 'CLV']])

# Encode binary categorical variables
data['Gender'] = data['Gender'].map({'Male': 0, 'Female': 1})
data['HasCreditCard'] = data['HasCreditCard'].map({'No': 0, 'Yes': 1})
data['IsActiveMember'] = data['IsActiveMember'].map({'No': 0, 'Yes': 1})

# One-hot encode remaining categorical features
data = pd.get_dummies(data, columns=['AccountType', 'Region'], drop_first=True)


# Verify Engineered Features
Preview newly created features to ensure correctness before modeling.


In [19]:
# Check engineered features
print("Preview of engineered features:")
print(data[['Balance_to_Salary', 'Balance_Tenure', 'Products_Active', 'TenureGroup', 'Segment']].head())

# Quick summary of feature distributions
print("\nFeature distributions:")
print(data[['Balance_to_Salary', 'Balance_Tenure', 'Products_Active', 'TenureGroup', 'Segment']].describe())


Preview of engineered features:
   Balance_to_Salary  Balance_Tenure Products_Active  TenureGroup  Segment
0           1.903731       737978.16          YesYes            1        3
1           0.172153        23364.19            NoNo            0        2
2           1.400170      1869229.84       YesYesYes            2        3
3           1.459903      1212439.92              No            3        1
4           0.139367       459911.34          YesYes            3        2

Feature distributions:
       Balance_to_Salary  Balance_Tenure  TenureGroup       Segment
count       50000.000000    5.000000e+04   50000.0000  50000.000000
mean            1.599591    1.188400e+06       1.9998      1.502120
std             1.621033    1.079441e+06       1.0950      1.114172
min             0.000038    0.000000e+00       0.0000      0.000000
25%             0.571412    2.921738e+05       1.0000      1.000000
50%             1.141401    8.770514e+05       2.0000      2.000000
75%             1.

# Define Features and Targets
Separate predictors from target variables (churn and CLV) and split into training/testing sets.


In [20]:
# Define Features and Target Variables
X = data.drop(['CustomerID', 'Exited', 'CLV'], axis=1)
y_churn = data['Exited']
y_clv = data['CLV']

# Split data into training and testing sets (Churn)
X_train_churn, X_test_churn, y_train_churn, y_test_churn = train_test_split(
    X, y_churn, test_size=0.2, random_state=42
)

# Split data into training and testing sets (CLV)
X_train_clv, X_test_clv, y_train_clv, y_test_clv = train_test_split(
    X, y_clv, test_size=0.2, random_state=42
)

# Feature Scaling
Standardize numeric features while preserving categorical variables for modeling.


In [21]:
# Feature Scaling (numeric variables only)

# Select numeric columns
numeric_cols = X.select_dtypes(include=[np.number]).columns
X_train_num = X_train[numeric_cols]
X_test_num = X_test[numeric_cols]

# Apply standard scaling
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_num),
    columns=numeric_cols,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test_num),
    columns=numeric_cols,
    index=X_test.index
)

# Recombine scaled numeric with categorical features
X_train_prepared = pd.concat([X_train_scaled, X_train.drop(columns=numeric_cols)], axis=1)
X_test_prepared = pd.concat([X_test_scaled, X_test.drop(columns=numeric_cols)], axis=1)


# Diagnostic Check for Non-Numeric Features
Identify columns with object/string values and preview unique entries to catch encoding issues.


In [23]:
# Diagnostic helper
non_numeric_cols = X_train_prepared.select_dtypes(include='object').columns

if len(non_numeric_cols) == 0:
    print("All features are numeric. Ready for modeling.")
else:
    print("Non-numeric columns detected:", list(non_numeric_cols))
    for col in non_numeric_cols:
        print(f"\nColumn: {col}")
        print("Sample unique values:", X_train_prepared[col].unique()[:10])


Non-numeric columns detected: ['Products_Active']

Column: Products_Active
Sample unique values: ['YesYesYes' 'YesYesYesYes' 'NoNoNo' 'YesYes' 'No' 'NoNoNoNo' 'NoNo' 'Yes']


# Auto-Fix for Binary Encoding Issues
Convert anomalous string values like "YesYesYes" into proper numeric encodings (0/1).


In [24]:
# Define binary columns that should only contain Yes/No values
binary_cols = ['HasCreditCard', 'IsActiveMember']

# Clean and encode binary columns
for col in binary_cols:
    if col in data.columns:
        data[col] = data[col].replace({
            'YesYesYes': 1,  # Fix anomaly
            'Yes': 1,
            'No': 0
        }).astype(int)

# Rebuild train/test prepared sets after cleaning
X = data.drop(['CustomerID', 'Exited', 'CLV'], axis=1)

# Re-split to ensure consistency
X_train, X_test, y_train_churn, y_test_churn = train_test_split(
    X, data['Exited'], test_size=0.2, random_state=42
)

# Scale numeric features again
numeric_cols = X.select_dtypes(include=[np.number]).columns
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train[numeric_cols]),
    columns=numeric_cols,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test[numeric_cols]),
    columns=numeric_cols,
    index=X_test.index
)

# Recombine scaled numeric with categorical features
X_train_prepared = pd.concat([X_train_scaled, X_train.drop(columns=numeric_cols)], axis=1)
X_test_prepared = pd.concat([X_test_scaled, X_test.drop(columns=numeric_cols)], axis=1)

print("Binary encoding fixed and features rebuilt. Ready for modeling.")


Binary encoding fixed and features rebuilt. Ready for modeling.


# Ensemble Model for Churn Prediction
Train a soft-voting ensemble (Random Forest, Gradient Boosting, Logistic Regression) and evaluate performance.


In [22]:
# Ensemble Model for Churn Prediction

# Initialize base models
rf = RandomForestClassifier(n_estimators=300, random_state=42)
gb = GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, random_state=42)
lr = LogisticRegression(max_iter=1000)

# Combine models using soft voting
voting_clf = VotingClassifier(
    estimators=[('rf', rf), ('gb', gb), ('lr', lr)], voting='soft'
)

# Train ensemble model
voting_clf.fit(X_train_prepared, y_train_churn)

# Make predictions
y_pred_churn = voting_clf.predict(X_test_prepared)

# Evaluate model performance
print('--- Ensemble Churn Prediction ---')
print('Accuracy:', round(accuracy_score(y_test_churn, y_pred_churn), 4))
print('ROC-AUC:', round(roc_auc_score(y_test_churn, voting_clf.predict_proba(X_test_prepared)[:, 1]), 4))
print(classification_report(y_test_churn, y_pred_churn))


ValueError: could not convert string to float: 'YesYesYes'

In [11]:
# CLV regression using Random Forest

# Initialize and train Random Forest regressor
rf_clv = RandomForestRegressor(n_estimators=300, random_state=42)
rf_clv.fit(X_train_scaled, y_train_clv)

# Make predictions
y_pred_clv = rf_clv.predict(X_test_scaled)

# Evaluate regression performance
print("\n--- CLV Regression Metrics ---")
print("RMSE:", round(np.sqrt(mean_squared_error(y_test_clv, y_pred_clv)), 2))
print("R2 Score:", round(r2_score(y_test_clv, y_pred_clv), 4))


--- CLV Regression Metrics ---
RMSE: 3824.25
R2 Score: 0.5188


In [12]:
# Calculate retention prioritization score

# Get predicted churn probabilities
y_proba_churn = voting_clf.predict_proba(X_test_scaled)[:,1]

# Combine churn risk with predicted CLV to prioritize retention
retention_priority = y_proba_churn * y_pred_clv

# Compile results into a DataFrame
results = pd.DataFrame({
    'CustomerID': data.loc[X_test.index, 'CustomerID'],
    'ActualChurn': y_test_churn,
    'PredictedChurn': y_pred_churn,
    'ChurnProbability': y_proba_churn,
    'ActualCLV': y_test_clv,
    'PredictedCLV': y_pred_clv,
    'RetentionPriorityScore': retention_priority
})

In [13]:
# Create and display results table

results = pd.DataFrame({
    'CustomerID': data.loc[X_test.index, 'CustomerID'],
    'ActualChurn': y_test_churn,
    'PredictedChurn': y_pred_churn,
    'ChurnProbability': y_proba_churn,
    'ActualCLV': y_test_clv,
    'PredictedCLV': y_pred_clv,
    'RetentionPriorityScore': retention_priority
})

# Show top 20 customers with retention priority
display(results.head(20))

print("Results table generated successfully")

,CustomerID,ActualChurn,PredictedChurn,ChurnProbability,ActualCLV,PredictedCLV,RetentionPriorityScore
33553,33554,0,0,0.236320,3658.58,5562.092533,1314.436357
9427,9428,0,0,0.149429,1472.45,1829.314500,273.353309
199,200,0,0,0.243385,4058.88,4272.836733,1039.946351
12447,12448,1,0,0.161795,1538.06,1645.460667,266.227720
39489,39490,0,0,0.156149,6984.44,13423.757133,2096.104146
42724,42725,0,0,0.165146,5663.51,4219.008400,696.750506
10822,10823,0,0,0.186547,20282.34,11894.727700,2218.925279
49498,49499,0,0,0.157038,2673.13,2700.784833,424.125065
4144,4145,1,0,0.330104,6031.41,5168.527133,1706.153931
36958,36959,0,0,0.163363,9323.74,7364.275600,1203.051121


Results table generated successfully


In [14]:
# SHAP explainability for model insights

print("\nRunning SHAP...")

# Convert scaled training and testing data to DataFrames for SHAP
X_train_df = pd.DataFrame(X_train_scaled, columns=numeric_cols)
X_test_df = pd.DataFrame(X_test_scaled, columns=numeric_cols)

# Select a random sample from the test set for explainability
X_sample = X_test_df.sample(n=50, random_state=42)

# CLV Explainability using SHAP
explainer_clv = shap.TreeExplainer(rf_clv)
shap_values_clv = explainer_clv.shap_values(X_sample)
shap.summary_plot(shap_values_clv, X_sample)

# Churn Explainability model
rf_explain = RandomForestClassifier(n_estimators=50, random_state=42)
rf_explain.fit(X_train_scaled, y_train_churn)

explainer_churn = shap.TreeExplainer(rf_explain)
shap_values_churn = explainer_churn.shap_values(X_sample)

# Handle class-specific SHAP values (for binary classification)
if isinstance(shap_values_churn, list):
    shap_matrix = shap_values_churn[1]  # Class 1 (churn)
else:
    shap_matrix = shap_values_churn

# Ensure SHAP matrix and feature set align
if shap_matrix.shape[1] > X_sample.shape[1]:
    shap_matrix = shap_matrix[:, :X_sample.shape[1]]

# Churn SHAP summary plot
shap.summary_plot(shap_matrix, X_sample)


Running SHAP...


NameError: name 'shap' is not defined

In [ ]:

# Visualizations

# Churn distribution plot
sns.countplot(x='Exited', data=data)
plt.title("Customer Churn Distribution")
plt.show()

# Customer CLV (Customer Lifetime Value) distribution plot
sns.histplot(data['CLV'], bins=50, kde=True)
plt.title("Customer CLV Distribution")
plt.show()

# Feature correlation heatmap
plt.figure(figsize=(12,10))
sns.heatmap(data.corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Feature Correlations")
plt.show()

In [ ]:
# ADVANCED VISUALIZATIONS

import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

# --- 1. Churn vs CLV Scatter Plot ---
fig_churn_clv = px.scatter(
    data, 
    x='CLV', 
    y='Exited', 
    color='Exited', 
    opacity=0.7,
    title='Churn vs Customer Lifetime Value (CLV)',
    labels={'Exited': 'Churn (0=No, 1=Yes)', 'CLV': 'Customer Lifetime Value'}
)
fig_churn_clv.update_layout(yaxis=dict(dtick=1))  # Ensure y-axis is discrete (0/1)
fig_churn_clv.show()

# --- 2. Top Retention Priorities Bar Plot ---
fig_priority = px.bar(
    results.sort_values(by='RetentionPriorityScore', ascending=False).head(20),
    x='CustomerID', 
    y='RetentionPriorityScore',
    title='Top 20 Customers by Retention Priority Score',
    labels={'CustomerID': 'Customer ID', 'RetentionPriorityScore': 'Priority Score'}
)
fig_priority.show()


print("Advanced visualizations rendered successfully.")

In [ ]:
# Feature importance from the churn model

importances = rf_explain.feature_importances_
feat_imp = pd.DataFrame({'Feature': numeric_cols, 'Importance': importances}).sort_values(by='Importance', ascending=False)

# Plot feature importance using a horizontal bar plot
sns.barplot(x='Importance', y='Feature', data=feat_imp)
plt.title("Feature Importance (Churn Model)")
plt.show()

In [ ]:
# Summary of Analysis

print("Analysis Summary:")
print("1. Data was loaded and cleaned, handling missing values and encoding categorical variables.")
print("2. Advanced features were engineered, including ratios and tenure groups.")
print("3. The dataset was split into train/test sets, and numeric features were scaled.")
print("4. An ensemble model (Random Forest, Gradient Boosting, and Logistic Regression) was used for churn prediction; a Random Forest was used for CLV regression.")
print("5. Retention priority scores were computed by combining churn probability and predicted CLV.")
print("6. SHAP values provided insights into feature importance for both churn and CLV models.")
print("7. Advanced visualizations, including Plotly scatter and bar charts, were used to explore churn, CLV, and customer retention priorities.")

In [ ]:
# Save results to a CSV file

results.to_csv('../data/bank_customer_results.csv', index=False)
print("Results saved to 'bank_customer_results.csv'.")